In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

In [3]:
sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]
labels = [1]*15 + [0]*15
labels = np.array(labels)

In [4]:
vocab_size = 2000
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
seqs = tokenizer.texts_to_sequences(sentences)
maxlen = max(len(seq) for seq in seqs)
X = pad_sequences(seqs, maxlen=maxlen, padding="post")
y = labels

In [5]:
embed_dim = 16
rnn_units = 8


In [6]:
inp = Input(shape=(maxlen,), dtype="int32", name="input")
x = Embedding(input_dim=vocab_size, output_dim=embed_dim, name="embed")(inp)
rnn = SimpleRNN(units=rnn_units, return_sequences=True, return_state=True, name="simple_rnn")
_, x_last = rnn(x) # Unpack the tuple, taking the last state
out = Dense(1, activation="sigmoid", name="out" )(x_last)
model = Model(inputs=inp, outputs=out)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 5)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embed (Embedding)               │ (None, 5, 16)          │        32,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ [(None, 5, 8), (None,  │           200 │
│                                 │ 8)]                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ out (Dense)                     │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.fit(X, y, epochs=25, batch_size=8,verbose=1);

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.5333 - loss: 0.6917
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7333 - loss: 0.6742
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8000 - loss: 0.6597
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8333 - loss: 0.6436
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8333 - loss: 0.6290
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8333 - loss: 0.6133
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8667 - loss: 0.5950
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9000 - loss: 0.5765
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9667 - loss: 0.5574
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.5370
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 0.5141
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.4924
E

In [10]:
intermediate_model = Model(inputs=inp, outputs=rnn(x))

# Predict to get the pre-timestamps and hidden states
pre_timestamps, hidden_states = intermediate_model.predict(X)

print("Shape of pre-timestamps (outputs at each timestep):", pre_timestamps.shape)
print("Shape of hidden states (last state):", hidden_states.shape)

# You can now analyze or visualize these arrays.
# For example, to see the pre-timestamps for the first sentence:
# print("Pre-timestamps for the first sentence:\n", pre_timestamps[0])
# To see the hidden state for the first sentence:
# print("Hidden state for the first sentence:\n", hidden_states[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
Shape of pre-timestamps (outputs at each timestep): (30, 5, 8)
Shape of hidden states (last state): (30, 8)


In [11]:
from tensorflow.keras.layers import SimpleRNN as SRNN
seq_inp = Input(shape=(maxlen,), dtype='int32')
seq_emb = model.get_layer('embed') (seq_inp) # reuse trained embedding
#Create a RNN with same units, but return_sequences=True
rnn_seq= SRNN(units=rnn_units, return_sequences=True, name='rnn_seq')
# The rnn_seq.build() line is removed as it's causing the error and is not needed.
seq_hidden = rnn_seq(seq_emb)

In [12]:
inspect_model = Model(inputs=seq_inp, outputs=seq_hidden)

# Predict to get the sequence of hidden states
sequence_hidden_states = inspect_model.predict(X)

print("Shape of sequence hidden states:", sequence_hidden_states.shape)

# You can now analyze or visualize these sequence hidden states.
# For example, to see the sequence of hidden states for the first sentence:
# print("Sequence hidden states for the first sentence:\n", sequence_hidden_states[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 230ms/step
Shape of sequence hidden states: (30, 5, 8)


In [13]:
rnn_weights = rnn_seq.get_weights()

print("RNN Weights (Input to Hidden, Recurrent to Hidden, Bias):")
for i, w in enumerate(rnn_weights):
    print(f"Weight {i+1} shape: {w.shape}")
    # print(f"Weight {i+1}:\n{w}\n") # Uncomment to see the actual weight values

RNN Weights (Input to Hidden, Recurrent to Hidden, Bias):
Weight 1 shape: (16, 8)
Weight 2 shape: (8, 8)
Weight 3 shape: (8,)
